In [8]:
%pip install sentence-transformers langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1.setting
# load PDF
PDF_PATH = "data/2306.08045v2.pdf" 
DB_PATH = "faiss_index"

# 2. load local module
print("start Embedding ...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 3. slice
if not os.path.exists(PDF_PATH):
    print(f"can't find {PDF_PATH}")
else:
    print("reading...")
    loader = PyPDFLoader(PDF_PATH)
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    splits = splitter.split_documents(docs)

    # 4. create DB
    if os.path.exists(DB_PATH):
        print("load to DB")
        vectorstore = FAISS.load_local(DB_PATH, embeddings, allow_dangerous_deserialization=True)
    else:
        print("create Faiss DB...")
        vectorstore = FAISS.from_documents(splits, embeddings)
        vectorstore.save_local(DB_PATH)
        print("save ok")

    # 5. test RAG 
    query = "What is the core contribution of this paper?"
    results = vectorstore.similarity_search(query, k=2)
    print("\n Search result：", results[0].page_content[:200])

d:\AI\RAG_project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


start Embedding ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1057.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


reading...
load to DB

 Search result： | N (p)| × Dval. The modules T i
enc and T i
dec gather contextual
information as follows:
[T (m)]p
+
= att(Q⊺
p ⊕Aque
p , KN (p) +Akey
p , VN (p) +Aval
p ) , (3)
with
+
= a residual connection [20], 


In [13]:
context = results[0].page_content
prompt = f"請用簡單的語言解釋以下這段數學公式的物理意義：\n{context}"
# 然後呼叫你的 LLM (如 Ollama) 來回答

In [14]:
from langchain_community.llms import Ollama

# 1. 初始化你的本地 LLM (請確保你有安裝 Ollama 並且在後台運行)
# 這裡假設你已經跑過 'ollama run llama3'
llm = Ollama(model="llama3")

# 2. 發送 prompt 給模型
print("🤖 AI 正在思考中...")
response = llm.invoke(prompt)

# 3. 輸出結果
print("\n📝 AI 的回答：")
print(response)

C:\Users\tony\AppData\Local\Temp\ipykernel_17152\7272482.py:5: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3")


🤖 AI 正在思考中...


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [WinError 10061] 無法連線，因為目標電腦拒絕連線。"))